# Data Intelligence Platform & Compute

## What's covered

- The problem the Data Intelligence Platform solves — why a lakehouse beat the warehouse-plus-lake split
- Architecture: control plane vs. compute plane, and what lives where
- The four foundation layers — cloud storage, Delta Lake, Spark + Photon, Unity Catalog
- Workspace assets you touch as a data engineer — notebooks, Git folders, Volumes, Lakeflow Jobs, DBSQL
- Compute services in detail — all-purpose clusters, job clusters, SQL warehouses (classic / pro / serverless), serverless compute for notebooks and jobs
- Choosing the right compute for a workload — the four scenarios the exam keeps testing
- Cost model intuition — DBUs, on-demand vs. spot, autoscaling, auto-termination
- Databricks Runtime (DBR LTS) and Photon — what they buy you
- Where Lakeflow Connect, Lakeflow Spark Declarative Pipelines, and Mosaic AI fit (preview; deeper in notebooks 03, 05, and beyond)

## Reference domain

Every concept in this course is applied to the **Fintech consumer bank** from `../Fintech` — four verticals (**Cards**, **Loans**, **Accounts**, **Payments**) sharing a common **Customers** dimension. The bank wants a single platform that ingests raw events from all four sources, curates them into bronze / silver / gold layers, and serves dashboards, ML, and ad-hoc SQL to analysts. This notebook is about the platform we'll build everything on top of.

## The problem — warehouse vs. lake, and why we needed both fused

Before the lakehouse, the bank's data team would have run **two parallel stacks**.

A **data warehouse** (Snowflake, Redshift, Synapse) gave them fast SQL, ACID transactions, and strict schemas — perfect for the finance dashboard reporting yesterday's card volume. But it was expensive per terabyte, couldn't store raw JSON logs efficiently, and was locked away from data scientists who wanted Python and notebooks for fraud modelling.

A **data lake** (S3 + Parquet, Hive metastore) gave them cheap storage for raw events at any scale — fraud researchers loved it. But it had no transactions: a failed write could leave the loan-repayments table half-updated, and a reader hitting the table mid-write got a corrupted view. Schemas drifted. Governance was bolted on with three different tools.

Either choice forced the bank to copy data between systems, reconcile two sources of truth, and run two security models. The fraud team's model and the CFO's dashboard would disagree on what "today's transactions" meant.

The **lakehouse** fuses the two: cheap object storage at the bottom, an open table format (Delta Lake) on top that adds ACID and schema enforcement, one governance layer (Unity Catalog) over everything, and one engine (Spark + Photon) that serves both SQL analytics and Python ML. The bank stores raw card transactions once, curates them once, and serves the same table to the dashboard, the fraud model, and the regulator.

## What the Data Intelligence Platform actually is

The **Databricks Data Intelligence Platform** is the lakehouse, plus an AI-native layer on top. Three sentences worth memorising:

- It is **open at the storage layer** — your data lives in your own cloud account (S3, ADLS, or GCS) as Delta tables, an open format Databricks does not lock you into.
- It is **unified at the engine layer** — Spark (with Photon as the C++ vectorised accelerator) handles batch ETL, streaming, SQL analytics, and ML training as one runtime.
- It is **governed at the catalog layer** — Unity Catalog is the single source of truth for permissions, lineage, discovery, and audit across all workloads.

Sitting on top of that core, the platform layers in **AI-aware** features: a built-in assistant, intelligent search, natural-language interfaces to data, and Mosaic AI for serving models. The exam mostly cares about the lakehouse core — the AI layer is supporting context.

## Architecture — control plane vs. compute plane

When you sign in to a Databricks workspace, you are talking to **two distinct planes**.

The **control plane** is the part Databricks operates in *its* cloud account. It serves the workspace UI, holds notebook source, stores job definitions, runs the cluster manager that provisions VMs, and houses Unity Catalog metadata. Your *data* never sits in the control plane.

The **compute plane** is where your queries actually run, and it lives in **your** cloud account (or a Databricks-managed serverless tenant). Cluster VMs, executor JVMs, and the Spark drivers themselves run here. The data they read sits in your S3 / ADLS / GCS buckets, also in your account.

```text
        Databricks control plane                Your cloud account
 (Databricks-managed AWS/Azure/GCP)             (compute plane)
 ┌────────────────────────────────┐        ┌──────────────────────────┐
 │  Workspace UI                  │        │  Cluster VMs             │
 │  Notebook + job metadata       │ ─────► │  ┌───────┐  ┌─────────┐  │
 │  Unity Catalog metadata        │  REST  │  │Driver │  │Executors│  │
 │  Cluster manager (provisions)  │ ◄───── │  └───────┘  └─────────┘  │
 │  Notebook results metadata     │        │                          │
 └────────────────────────────────┘        │  Reads/writes to ──┐     │
                                           └─────────────────────┼────┘
                                                                 ▼
                                              ┌──────────────────────┐
                                              │  S3 / ADLS / GCS     │
                                              │  Delta tables + raw  │
                                              └──────────────────────┘
```

Why this split matters for the exam: **data residency, security boundaries, and cost** all line up with the compute plane. The bank's regulator cares that customer PII never leaves their AWS account — and with classic compute, it doesn't.

**Serverless compute** is the exception: those VMs run in a Databricks-managed compute plane, not yours. You pay only for what you use, you start in seconds, but you trade away the "VMs in my account" property.

## The foundation stack — four layers

Read from the bottom up — each layer is built on the one below.

```text
 ┌──────────────────────────────────────────────────────────┐
 │  Unity Catalog                                            │  Governance
 │  catalog.schema.table  +  GRANT/REVOKE  +  lineage        │  (notebook 09)
 ├──────────────────────────────────────────────────────────┤
 │  Spark + Photon                                           │  Engine
 │  one runtime: batch / streaming / SQL / ML                │  (notebook 04)
 ├──────────────────────────────────────────────────────────┤
 │  Delta Lake                                               │  Storage format
 │  ACID, time travel, schema enforcement, MERGE             │  (notebook 02)
 ├──────────────────────────────────────────────────────────┤
 │  Cloud object storage                                     │  Bytes
 │  S3 / ADLS / GCS  —  Parquet files + _delta_log           │  (your account)
 └──────────────────────────────────────────────────────────┘
```

**Cloud storage** is just bytes on disk — Parquet files plus a transaction log directory. Cheap, durable, infinitely scalable, but with no transactional guarantees on its own.

**Delta Lake** turns those raw files into proper tables. The `_delta_log` directory holds an ordered sequence of JSON commit files; readers compose them to see a consistent snapshot. This is where ACID transactions, time travel, schema enforcement, and `MERGE INTO` come from. Covered in depth in notebook 02.

**Spark + Photon** is the compute engine. Open-source Spark does the planning; Photon — Databricks' proprietary C++ vectorised execution engine — accelerates the hot operators (filter, aggregate, join) on supported runtimes. Photon is enabled per-cluster and is the default on serverless and SQL warehouses.

**Unity Catalog** is the governance layer that ties it all together — every table, view, volume, model, and function lives under `catalog.schema.object`, and every access goes through a single permission model. Covered in depth in notebook 09.

## The workspace — assets a data engineer touches daily

A **workspace** is the unit of organisation in Databricks — typically one per environment (dev / test / prod) per business unit. Inside it, you'll work with:

| Asset | What it is | Where it lives |
|---|---|---|
| **Notebooks** | Interactive Python / SQL / Scala / R documents | Workspace files or Git Folders |
| **Databricks Git Folders** | A workspace folder backed by a Git remote — branch, commit, push, PR from the UI | Workspace, synced to GitHub/Azure DevOps/GitLab |
| **Lakeflow Jobs** | Orchestrated DAGs of tasks (notebook, SQL, pipeline, dashboard) | Workspace; runs on a cluster (notebook 06) |
| **Lakeflow Spark Declarative Pipelines** | Declarative ETL — formerly known as **Delta Live Tables (DLT)** | Workspace; runs on dedicated pipeline compute (notebook 05) |
| **DBSQL** | SQL editor, queries, dashboards, alerts | Workspace; runs on SQL warehouses |
| **Volumes (UC)** | Governed file storage inside Unity Catalog — for unstructured / non-tabular data | `catalog.schema.volume_name` |
| **Models** | MLflow models registered in Unity Catalog | `catalog.schema.model_name` |
| **Secrets** | Scoped key-value store for credentials | Secret scopes; read via `dbutils.secrets` |

**Rename watch** — three names have changed recently and the exam tests current naming:

- **Lakeflow Jobs** — formerly Databricks Workflows / Jobs.
- **Lakeflow Spark Declarative Pipelines** — formerly Delta Live Tables (DLT).
- **Databricks Git Folders** — formerly Databricks Repos.

## The three-level namespace — `catalog.schema.object`

Unity Catalog organises every governed object into a strict three-level hierarchy. The bank's data ends up looking like this:

```text
 metastore (one per region)
  │
  ├── catalog: fintech_dev      ← environment
  │    ├── schema: bronze       ← raw landed data
  │    │    ├── card_transactions_raw
    │   │    └── loan_repayments_raw
  │    ├── schema: silver       ← cleaned, conformed
  │    │    ├── card_transactions
  │    │    └── customers
  │    └── schema: gold         ← business-ready
  │         ├── customer_360
  │         └── daily_card_volume
  ├── catalog: fintech_prod
  └── catalog: ml_models
       └── schema: fraud
            └── model: card_fraud_v3
```

**Why three levels matter for compute selection:** permission scopes follow the namespace. The fraud team's SQL warehouse can be granted `USE CATALOG` on `fintech_prod` and `SELECT` only on `gold` — meaning the same physical warehouse can safely serve analysts without exposing bronze.

The legacy `hive_metastore` two-level form (`schema.table`) is still visible in old workspaces, but every new table in this course goes into Unity Catalog. Treat `hive_metastore.*` as something you migrate **off**, not onto.

# Compute services

Compute is the part of the platform you pay for by the second, and it is the area the exam tests most heavily in Section 1. Pick the wrong one and you either burn money, throttle users, or fail SLAs.

There are essentially **four families** of compute. Each has a workload it is built for.

| Family | Built for | Pricing unit | Startup |
|---|---|---|---|
| **All-purpose cluster** | Interactive, multi-user notebook development | DBU/hour (all-purpose rate) | ~3–8 min |
| **Job cluster** | Scheduled / triggered Lakeflow Jobs | DBU/hour (jobs rate — cheaper) | ~3–5 min per run |
| **SQL warehouse** (classic / pro / serverless) | BI dashboards and ad-hoc SQL | DBU/hour (SQL rate) | classic: minutes; serverless: seconds |
| **Serverless compute for notebooks / jobs** | Fast-starting notebook + job workloads, no cluster mgmt | DBU/hour (serverless rate) | seconds |

**DBU** = Databricks Unit, a per-second compute consumption metric. The dollar rate per DBU depends on the workload family (all-purpose is the most expensive, jobs the cheapest) and the cloud SKU. On top of DBUs you also pay your cloud provider for the underlying VMs and storage — except on serverless, where Databricks bundles everything into one DBU rate.

## All-purpose clusters — interactive, multi-user development

An **all-purpose cluster** is a long-lived cluster you create, attach a notebook to, and share with teammates. It stays up between commands so you don't pay startup time on every cell.

**Use when:** a data engineer is iterating on a transformation, a small team is sharing a development cluster, or you need to attach multiple notebooks at once.

**Don't use when:** running scheduled production ETL — you'd pay the more expensive all-purpose DBU rate for work a job cluster could do cheaper, and you'd risk concurrent users interfering with the job.

**Key knobs:**

- **Mode** — *Single user* (recommended for UC) or *Shared* (multi-user, isolates users via Unity Catalog).
- **Autoscaling** — set a min and max worker count; the cluster scales workers as load changes.
- **Auto-termination** — *required* to control cost. Idle for N minutes → cluster shuts down. 30–60 minutes is typical for dev.
- **DBR version** — pick a Long-Term Support (LTS) runtime in production for stability.

## Job clusters — scheduled, ephemeral, cheaper

A **job cluster** is created by Lakeflow Jobs at the start of a run and torn down at the end. Each scheduled execution of the bank's nightly card-transactions ETL spins up a fresh cluster, runs the notebook, writes the output, and dies.

**Use when:** any production scheduled or triggered workload. This is the default for Lakeflow Jobs tasks.

**Why it's the right default:**

- **Cheaper DBU rate** than all-purpose — Databricks prices jobs compute lower because it's non-interactive.
- **Isolation** — each run gets a clean state; no leftover variables, no library conflicts from a previous run.
- **Right-sized per task** — different tasks in the same job can use different cluster configs.
- **Auto-cleanup** — you cannot forget to shut it down. It dies when the run ends.

**Trade-off:** ~3–5 minutes of startup per run. For sub-minute scheduling cadence, use serverless compute instead.

## SQL warehouses — BI and ad-hoc SQL

A **SQL warehouse** is a compute target specialised for SQL queries — concurrent, low-latency, Photon-accelerated. Three flavours, in order of capability:

| Warehouse type | Compute plane | Startup | When to pick |
|---|---|---|---|
| **Classic** | Your cloud account | ~minutes | Legacy / data residency requires VMs in your account |
| **Pro** | Your cloud account | ~minutes | Advanced features (Predictive I/O, geospatial) in your account |
| **Serverless** | Databricks-managed | ~seconds | The default for new workloads — instant scale, lowest TCO |

All three speak ANSI SQL, are Photon-on, and use the same SQL editor / dashboards / alerts surface. The difference is where the VMs run and how fast they start.

**Concurrency model — and the exam-classic question pattern:** when business analysts run ad-hoc SQL throughout the day, you want a warehouse that **autoscales clusters** to handle concurrent users, starts fast, and shuts down when idle. That is a **serverless SQL warehouse**. The older "high-concurrency cluster" answer (still on some sample exams) is its predecessor — the modern equivalent is a serverless SQL warehouse with autoscaling enabled.

**Multi-cluster autoscaling** (Pro / Serverless): you set min and max *clusters* (not just workers). As query queue depth grows, more clusters spin up; as it falls, they spin down. Sized for *concurrency*, not for the size of any single query.

## Serverless compute for notebooks and jobs

Beyond SQL warehouses, there is now **serverless compute** for notebook interactive use and for Lakeflow Jobs tasks. Same idea — Databricks-managed VMs, second-scale startup, no cluster management — extended from SQL to the full notebook surface.

**Use when:** development with fast iteration, lightweight scheduled tasks, or anywhere a 3-minute job-cluster startup is too long for the workload.

**Trade-offs to know:**

- **No init scripts, no arbitrary cluster libraries** — you install Python packages via `%pip` notebook-scoped, which is what most workloads want anyway.
- **No direct cluster customisation** — instance type, Spark configs, network are managed for you.
- **Workloads that need very specific JARs, GPUs, or custom Spark configs** still need classic compute.

The rule of thumb the exam rewards: **prefer serverless unless something explicitly forces classic** (data residency in your VPC, GPU workloads, init scripts, exotic libraries).

## Choosing compute — the four-scenario cheat sheet

The Section 1 questions almost always pose a *workload* and ask you to pick the compute. Map each scenario to the compute family.

| Scenario | Right answer |
|---|---|
| **Analysts run ad-hoc SQL all day on curated tables, fast startup, many users, cost-conscious** | Serverless SQL warehouse with autoscaling |
| **Nightly scheduled ETL — bronze → silver → gold for the bank's loans vertical** | Job cluster (or serverless jobs compute for short tasks) attached to a Lakeflow Jobs task |
| **A single engineer prototyping a new feature in a notebook for the next 3 hours** | Single-user all-purpose cluster (or serverless for notebooks) with auto-termination |
| **A POC running entirely on small data on the driver — no distribution needed** | Single-node cluster (driver only, no workers) |

Three traps the exam plants:

1. **"Add more workers to the all-purpose cluster"** when the real answer is "use a serverless SQL warehouse for SQL workloads" — adding workers does not fix the wrong compute family.
2. **"All-purpose cluster for scheduled ETL"** — works but burns money at the higher DBU rate and risks cross-job interference.
3. **"High-concurrency cluster"** — a legacy concept; the modern answer is a serverless SQL warehouse.

## Cost model — DBUs, autoscaling, auto-termination

Three levers control compute cost.

**DBU rate by workload family** (relative, exact $ varies by cloud / region):

```text
  Jobs Light  <  Jobs Compute  <  SQL Warehouse  <  All-Purpose  <  Serverless (managed convenience)
```

Picking the right family alone can change a workload's bill by 2–3×.

**Spot vs. on-demand workers** (classic compute only): spot instances are 60–90% cheaper but can be evicted. The Spark driver should always be on-demand (losing it kills the job); workers can mix spot + on-demand for ETL where one or two evicted tasks are tolerable.

**Autoscaling** lets the cluster adjust worker count to load — set realistic `min` and `max`. Too high a max wastes money on partially-utilised workers; too low a max throttles the job.

**Auto-termination** is the single biggest cost control on all-purpose clusters. A 30-minute idle timeout means a forgotten notebook costs at most 30 minutes of compute, not a weekend's worth. Job clusters and serverless terminate automatically by design.

## Databricks Runtime (DBR) and Photon

Every cluster runs a **Databricks Runtime** — Databricks' tuned distribution of Spark, plus Delta Lake, language runtimes, and dozens of pre-installed libraries. Pick a runtime by **major version + flavour**.

- **DBR `n.x` LTS** — Long-Term Support runtime. Bug fixes for 2 years. **Use for production**.
- **DBR `n.x`** — non-LTS, newer features, shorter support window. Use for dev / early features.
- **DBR ML** — adds TensorFlow, PyTorch, scikit-learn, MLflow. For ML workloads.
- **DBR Photon** — adds the Photon vectorised engine (default on serverless and SQL warehouses).

**Photon** is a C++ rewrite of Spark's hot execution path that vectorises operators and uses cache-aware columnar processing. It can be 2–10× faster on typical SQL aggregations and joins, charged at a higher DBU rate that usually still nets a lower total bill. On classic clusters it's a checkbox; on SQL warehouses and serverless it's on by default.

## Mapping the Fintech bank's workloads to the platform

Putting it all together — here is how the bank's needs from `../Fintech` line up with platform pieces.

| Bank requirement | Platform piece | Compute |
|---|---|---|
| Ingest daily snapshots from Cards / Loans / Accounts / Payments source systems | Lakeflow Connect, Auto Loader, COPY INTO (notebook 03) | Job cluster (scheduled) |
| Curate bronze → silver → gold with PySpark / SQL | Spark + Photon over Delta tables (notebook 04, 05) | Job cluster (Lakeflow Jobs task) |
| Declarative ETL with data quality expectations | Lakeflow Spark Declarative Pipelines (notebook 05) | Dedicated pipeline compute |
| Real-time fraud signals on card transactions | Structured Streaming over Delta (notebook 05) | Job cluster with continuous trigger |
| Analyst ad-hoc SQL on `customer_360` and `daily_card_volume` | DBSQL queries + dashboards (notebook 06) | Serverless SQL warehouse |
| Restrict PAN / Aadhaar columns to compliance team only | Unity Catalog row filters + column masks (notebook 09) | Governs all of the above |
| Promote pipelines dev → test → prod | Automation Bundles + Databricks CLI (notebook 07) | Same compute, different targets |

Notice the same compute *families* show up over and over — picking the right one is a small number of well-defined choices, not a per-workload reinvention.

## Hands-on — exploring the platform

The code cells below illustrate what you'd run in a **Databricks notebook attached to a cluster or SQL warehouse**. They assume:

- A workspace with Unity Catalog enabled
- Permission to create a catalog (or use an existing dev catalog)
- A cluster or SQL warehouse running Databricks Runtime 14.x+

If you're reading locally, treat them as the recipes you'd paste into a workspace — Databricks Community Edition or a trial workspace is the recommended hands-on environment.

Because the workspace already exposes a `SparkSession` named `spark`, you do **not** create one yourself the way you would locally — the cluster does that for you when the notebook attaches.

In [ ]:
# In a Databricks notebook, `spark` is pre-created. Confirm the runtime.
print("Spark version :", spark.version)
print("DBR version   :", spark.conf.get("spark.databricks.clusterUsageTags.sparkVersion", "(not on Databricks)"))
print("Photon on?    :", spark.conf.get("spark.databricks.photon.enabled", "(unknown)"))

In [ ]:
# List catalogs visible to you. In a fresh workspace you'll see `main`, `samples`, and `system`.
spark.sql("SHOW CATALOGS").show(truncate=False)

In [ ]:
# Stand up the Fintech catalog and the medallion schemas we'll use throughout the course.
# These are idempotent — safe to re-run.
spark.sql("CREATE CATALOG IF NOT EXISTS fintech_dev")
spark.sql("USE CATALOG fintech_dev")

for layer in ("bronze", "silver", "gold"):
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {layer}")

spark.sql("SHOW SCHEMAS IN fintech_dev").show(truncate=False)

In [ ]:
# A small managed Delta table in the silver schema — illustrates the three-level namespace.
# `customer_id` is STRING throughout, matching the Fintech data model from ../Fintech.
spark.sql("""
    CREATE TABLE IF NOT EXISTS fintech_dev.silver.customers (
        customer_id   STRING,
        full_name     STRING,
        email         STRING,
        city          STRING,
        country       STRING,
        created_at    TIMESTAMP
    )
    USING DELTA
    COMMENT 'Conformed customer dimension — shared across Cards / Loans / Accounts / Payments'
""")

# Insert a few rows mirroring the Fintech sample data.
spark.sql("""
    INSERT INTO fintech_dev.silver.customers VALUES
      ('C001', 'Priya Sharma', 'priya@example.com', 'Mumbai',    'IN', TIMESTAMP '2022-01-10 00:00:00'),
      ('C002', 'Arjun Mehta',  'arjun@example.com', 'Bangalore', 'IN', TIMESTAMP '2021-06-05 00:00:00'),
      ('C003', 'Sneha Kapoor', 'sneha@example.com', 'Delhi',     'IN', TIMESTAMP '2023-03-20 00:00:00')
""")

spark.sql("SELECT * FROM fintech_dev.silver.customers").show(truncate=False)

In [ ]:
# DESCRIBE EXTENDED shows storage location, owner, format — proof the table is governed by Unity Catalog
# and stored as a managed Delta table.
spark.sql("DESCRIBE EXTENDED fintech_dev.silver.customers").show(truncate=False, n=50)

## A first taste — managed vs. external tables

The table you just created is a **managed table**: Unity Catalog owns both the metadata *and* the underlying files. Drop it, and the files are deleted (after a retention period).

An **external table** points at files in a location you already own — typically a path inside a Unity Catalog external location. Dropping it deletes only the metadata; the files stay.

Choosing between the two is a Section 7 topic and gets a dedicated walkthrough in notebook 09. For now, remember the default: **prefer managed**, because Unity Catalog can apply Predictive Optimization, Liquid Clustering, and lifecycle management to data it controls end to end. External tables are for files you must share with non-Databricks systems.

## Section 1 self-check

Five questions in the style the exam uses. Answers at the bottom.

**1.** A team of 12 business analysts at the bank runs ad-hoc SQL on the `gold.customer_360` table all day. The team needs fast startup, support for concurrent users, and tight cost control. Which compute is best?

- A. An all-purpose cluster with autoscaling
- B. A job cluster scheduled to start every morning
- C. A serverless SQL warehouse with multi-cluster autoscaling
- D. A single-node cluster

**2.** The bank's nightly loans ETL has run on a shared all-purpose cluster for a year. Cost is rising and one engineer's interactive work occasionally collides with the job. What is the recommended change?

- A. Increase auto-termination to keep the cluster warm
- B. Move the ETL to a Lakeflow Jobs task running on its own job cluster
- C. Switch the all-purpose cluster to single-user mode
- D. Disable autoscaling on the all-purpose cluster

**3.** Which statement about the control plane and the compute plane is correct?

- A. Customer data flows through the control plane
- B. Classic compute VMs run in the customer's cloud account; the control plane stores notebook + job metadata
- C. Unity Catalog metadata is stored in the compute plane
- D. Serverless compute runs in the customer's cloud account

**4.** A data engineer needs the rapid iteration of Delta Lake's ACID transactions and time travel, plus a single source of truth governed for both BI and AI workloads. Which strategy meets the requirements?

- A. DBFS CSV with manual file versioning
- B. Delta Lake with Unity Catalog
- C. Cloud object storage and ad-hoc SQL only
- D. In-memory DataFrames

**5.** Which item is **not** a current Databricks product name?

- A. Lakeflow Jobs
- B. Lakeflow Spark Declarative Pipelines
- C. Databricks Git Folders
- D. Delta Live Tables

<details><summary>Show answers</summary>

1. **C** — serverless SQL warehouse autoscales for concurrency, starts in seconds, and shuts down when idle.
2. **B** — job clusters are cheaper per DBU and isolated per run.
3. **B** — classic compute runs in your cloud account; the control plane holds metadata, not customer data.
4. **B** — Delta Lake gives ACID + time travel; Unity Catalog gives the unified governance layer over BI and AI.
5. **D** — Delta Live Tables has been renamed to *Lakeflow Spark Declarative Pipelines*.

</details>

## Summary

- The **Data Intelligence Platform** is the lakehouse — open storage, one unified engine, one governance layer — with an AI-aware surface on top.
- The **control plane** (Databricks-managed) holds metadata; the **compute plane** (your cloud account, or Databricks-managed for serverless) runs queries against data in your cloud storage.
- Four foundation layers, bottom-up: **cloud storage → Delta Lake → Spark + Photon → Unity Catalog**.
- Workspace assets the exam expects you to recognise by their current names: **Lakeflow Jobs**, **Lakeflow Spark Declarative Pipelines**, **Databricks Git Folders**, **Volumes**, **DBSQL**.
- Every governed object lives at **`catalog.schema.object`** — the three-level namespace is how permissions, lineage, and discovery scope.
- **Four compute families**: all-purpose (dev), job (scheduled ETL), SQL warehouse (BI / ad-hoc SQL), serverless (fast-start managed). Pick by workload, not by habit.
- **Cost levers**: workload family DBU rate, autoscaling, auto-termination, spot vs. on-demand workers.
- **Photon** + **DBR LTS** are the production defaults.

Next up: **notebook 02 — Delta Lake & Unity Catalog Foundations** — what's actually in the `_delta_log`, how ACID works on object storage, and the managed-vs-external table choice in depth.